In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import meanderpy as mp
import cmocean

from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["figure.dpi"] = 1000

In [ ]:
SECONDS_PER_YEAR = 365.25 * 24 * 3600  # Seconds in one year; used to convert m/yr → m/s and years → seconds.
NIT = 1001  # Number of migration iterations (time steps). Total simulated time is NIT * DT_YEARS (years).
W = 100.0  # Channel width (m).
MAG = 1e-5  # Initial perturbation magnitude applied to one node in the perturbed run (m).
EXTENT = (0, 1000, -2500, 2500)  # Plotting / domain extent (xmin, xmax, ymin, ymax); not used directly in run_sim.
KL_M_PER_YR = 100.0  # Lateral migration rate scale kl (m/yr) used by meanderpy.
CRDIST = 2 * W  # Cutoff distance threshold (m). Controls when a cutoff is triggered (here: 2× channel width).

KV_M_PER_YR = 3.16e-5  # Vertical rate scale kv (m/yr) used by meanderpy (often very small).
DT_YEARS = 0.1  # Time step size (years) per iteration.

# Convert rates and timestep into SI units expected internally (m/s and s).
kl = KL_M_PER_YR / SECONDS_PER_YEAR  # Lateral migration rate in m/s.
kv = KV_M_PER_YR / SECONDS_PER_YEAR  # Vertical rate in m/s.
dt = DT_YEARS * SECONDS_PER_YEAR     # Time step in seconds.

def run_sim(pert, crdist):
    theta0 = 1.5          # Initial meander amplitude (radians) for the sinusoidal orientation angle θ(s).
    n_nodes = 1000        # Number of nodes used to discretize the centerline (fixed at initialization).
    total_length = 10000.0  # Total along-channel coordinate length used to parameterize the initial planform (m).
    lamb = 500.0          # Wavelength (m) of the initial sinusoidal θ(s).
    base_angle = 0.0      # Mean orientation angle (radians).

    # Arc-length-like coordinate along the initial centerline parameterization.
    s = np.linspace(0.0, total_length, n_nodes)

    # Initial orientation angle θ(s); sinusoidal perturbation about base_angle.
    theta = theta0 * np.sin(2.0 * np.pi * s / lamb) + base_angle

    # Integrate the heading θ(s) to construct initial (x, y) centerline coordinates.
    x = np.zeros_like(s)
    y = np.zeros_like(s)
    for i in range(1, n_nodes):
        ds = s[i] - s[i - 1]
        x[i] = x[i - 1] + ds * np.cos(theta[i])
        y[i] = y[i - 1] + ds * np.sin(theta[i])

    # Apply a localized perturbation to the initial condition (single-node y-offset at the midpoint).
    if pert != 0.0:
        mid = len(x) // 2
        y[mid] += pert

    # Flat bed elevation for all nodes (z = 0), and initialize meanderpy Channel / ChannelBelt objects.
    z = np.zeros_like(x)
    ch = mp.Channel(x, y, z, W, 1.0)
    chb = mp.ChannelBelt([ch], [], [0.0], [])

    # Simulation controls / parameters passed to meanderpy's migrate():
    saved_ts = 1            # Save every time step (1 = save all steps).
    deltas = 50.0           # Node spacing / resampling control used by meanderpy (m), depending on implementation.
    pad = 0                 # Padding for the channel-belt domain (0 = none).
    depths = np.ones(NIT)   # Flow depth time series (constant here).
    Cfs = 0.0065 * np.ones(NIT)  # Darcy–Weisbach friction factor time series (constant here).
    dens = 1000             # Fluid density (kg/m^3).

    # Timing / aggregation parameters (unused here but required by the API).
    t1 = t2 = t3 = 0
    aggr_factor = 0.0       # Aggregation factor (0 = off / none).

    # Run deterministic migration with the specified cutoff threshold crdist.
    chb.migrate(
        NIT, saved_ts, deltas, pad, crdist,
        depths, Cfs,
        kl, kv, dt,
        dens, t1, t2, t3, aggr_factor
    )
    return chb

# Reference and perturbed simulations (identical parameters; perturbed differs only by a single-node initial offset).
chb_ref = run_sim(0.0, CRDIST)
chb_pert = run_sim(MAG, CRDIST)

In [ ]:
times = np.arange(10, 1000, 1)  # Time indices of channel snapshots to plot.
cmap = plt.cm.Blues             # Sequential colormap for temporal evolution.
colors = cmap(np.linspace(0, 1.0, len(times)))  # Evenly spaced colors, avoiding extremely pale tones.

# --- Figure ---
fig, ax = plt.subplots(figsize=(5, 3))  # Compact figure sized for publication-style layout.

# Plot channel evolution as a family of blue curves (earlier → lighter, later → darker).
for t, c in zip(times, colors):
    ax.plot(
        chb_ref.channels[t].x,
        chb_ref.channels[t].y,
        lw=0.2,
        color=c,
        alpha=0.5
    )

# Highlight the final channel configuration in red.
ax.plot(
    chb_ref.channels[NIT - 1].x,
    chb_ref.channels[NIT - 1].y,
    lw=0.5,
    color='red',
    alpha=0.9,
    label="Unperturbed at t = 500"
)

# Axes formatting.
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal", "box")
ax.set_xlim([0, 5000])
ax.set_ylim([-1000, 1000])

plt.tight_layout()
plt.show()

In [ ]:
t_idx = 400
cell_size = 50.0  # Eulerian grid resolution (m per cell).

ch1 = chb_ref.channels[t_idx]   # Reference (unperturbed) channel at time index t_idx.
ch2 = chb_pert.channels[t_idx]  # Perturbed channel at the same time index.

# ---------------------------------------------
# 1. Define a common Eulerian grid (same extent/resolution for both channels)
# ---------------------------------------------
x_all = np.concatenate([ch1.x, ch2.x])  # Combined x-coordinates for auto-extents (optional).
y_all = np.concatenate([ch1.y, ch2.y])  # Combined y-coordinates for auto-extents (optional).

xmin = np.floor(x_all.min() / cell_size) * cell_size  # Snap min x to grid.
xmax = np.ceil(x_all.max() / cell_size) * cell_size   # Snap max x to grid.
ymin = np.floor(y_all.min() / cell_size) * cell_size  # Snap min y to grid.
ymax = np.ceil(y_all.max() / cell_size) * cell_size   # Snap max y to grid.

# Manually override extents to enforce a fixed domain across experiments/plots.
xmin, xmax = 0.0, 6000.0
ymin, ymax = -1000.0, 1000.0

cols = int((xmax - xmin) / cell_size)  # Number of grid columns (x-direction).
rows = int((ymax - ymin) / cell_size)  # Number of grid rows (y-direction).

# ---------------------------------------------
# 2. Rasterize a centerline onto the Eulerian grid (binary occupancy)
# ---------------------------------------------
def rasterize_channel(ch):
    g = np.zeros((rows, cols), dtype=bool)  # Eulerian state: True where the channel occupies a cell.

    # Densify each polyline segment so thin lines don't skip cells at coarse resolution.
    xs = np.linspace(ch.x[:-1], ch.x[1:], 10)
    ys = np.linspace(ch.y[:-1], ch.y[1:], 10)

    # Map (x, y) points to integer grid indices.
    col_idx = ((xs - xmin) / cell_size).astype(int)
    row_idx = ((ys - ymin) / cell_size).astype(int)

    # Keep only points that fall inside the grid.
    mask = (0 <= col_idx) & (col_idx < cols) & (0 <= row_idx) & (row_idx < rows)

    # Mark occupied cells.
    g[row_idx[mask], col_idx[mask]] = True
    return g

# ---------------------------------------------
# 3. Convert both channels into Eulerian occupancy grids
# ---------------------------------------------
G1 = rasterize_channel(ch1)  # Reference occupancy field.
G2 = rasterize_channel(ch2)  # Perturbed occupancy field.

# ---------------------------------------------
# 4. Visualize overlap/differences in Eulerian space
#    red    = reference only
#    blue   = perturbed only
#    purple = overlap
#    black  = empty
# ---------------------------------------------
img = np.zeros((rows, cols, 3), dtype=float)

img[G1 & ~G2, 0] = 1.0         # Reference-only cells.
img[G2 & ~G1, 2] = 1.0         # Perturbed-only cells.
img[G1 & G2] = [0.7, 0.0, 0.7] # Shared cells.

fig, ax = plt.subplots(1, 1, figsize=(5,3))
ax.imshow(img, origin='lower', extent=[xmin, xmax, ymin, ymax])
ax.set_aspect('equal')
ax.set_xlim([0, 5000])
ax.set_ylim([-1000, 1000])

plt.xlabel("x (m)")
plt.ylabel("y (m)")

plt.tight_layout()
plt.show()

In [ ]:
def get_log_diff(ch1, ch2, cell_size):
    """
    Compute the logarithm of the Eulerian Hamming distance between two channels.

    The two centerlines are rasterized onto a common fixed grid with resolution
    `cell_size`. The Hamming distance is defined as the number of grid cells whose
    occupancy differs between the two rasterized channels. The natural logarithm
    of this count is returned.
    """
    if ch1 is None or ch2 is None:
        return np.nan  # Undefined if either channel does not exist.

    # Determine a shared spatial extent enclosing both channels.
    x_all = np.concatenate([ch1.x, ch2.x])
    y_all = np.concatenate([ch1.y, ch2.y])

    xmin = x_all.min()
    xmax = x_all.max()
    ymin = y_all.min()
    ymax = y_all.max()

    # Snap bounds to the Eulerian grid.
    xmin_snap = np.floor(xmin / cell_size) * cell_size
    xmax_snap = np.ceil(xmax / cell_size) * cell_size
    ymin_snap = np.floor(ymin / cell_size) * cell_size
    ymax_snap = np.ceil(ymax / cell_size) * cell_size

    cols = int((xmax_snap - xmin_snap) / cell_size)
    rows = int((ymax_snap - ymin_snap) / cell_size)

    # Rasterize a single channel into a binary occupancy grid.
    def raster(c):
        g = np.zeros((rows, cols), dtype=bool)

        # Densify segments to avoid missing cells at finite resolution.
        xs = np.linspace(c.x[:-1], c.x[1:], 10)
        ys = np.linspace(c.y[:-1], c.y[1:], 10)

        col_idx = ((xs - xmin_snap) / cell_size).astype(int)
        row_idx = ((ys - ymin_snap) / cell_size).astype(int)

        mask = (0 <= col_idx) & (col_idx < cols) & (0 <= row_idx) & (row_idx < rows)
        g[row_idx[mask], col_idx[mask]] = True
        return g.astype(int)

    # Eulerian representations of the reference and perturbed channels.
    g1 = raster(ch1)
    g2 = raster(ch2)

    # Hamming distance: number of cells with different occupancy.
    diff = np.count_nonzero(g1 != g2)

    # Return log-distance (undefined for zero difference).
    return np.log(diff) if diff > 0 else np.nan


cell_size = 50.0  # Eulerian grid resolution (m).

step = 1
time_phys = np.arange(NIT) * DT_YEARS  # Physical time in years.
indices = np.arange(0, NIT, step)

# Log Hamming distance time series.
log_norms = np.array([
    get_log_diff(chb_ref.channels[t], chb_pert.channels[t], cell_size=cell_size)
    for t in indices
])

# Remove undefined values (e.g., zero difference).
mask = np.isfinite(log_norms)
t_data = time_phys[indices][mask]  # Time (years).
y_data = log_norms[mask]           # log Hamming distance (dimensionless).

fig, ax = plt.subplots(1, 1, figsize=(5,2))
plt.scatter(t_data, y_data, s=5, color='blue')
plt.xlabel("Time (years)")
plt.ylabel(r"$\log\, d_H$ (dimensionless)")
plt.xlim([0,100])
plt.tight_layout()
plt.show()